# 🧠 Mental Health NLP Analyzer: Detecting Depression and Anxiety from Social Media Text

---

> **Author:** Aastha Mahajan  
> **GitHub:** [aastha-m22/mental-health-nlp-analyzer](https://github.com/aastha-m22/mental-health-nlp-analyzer)  
> **Domain:** Natural Language Processing · Mental Health · Machine Learning  
> **Tools:** Python · scikit-learn · NLTK · TextBlob · WordCloud · XGBoost  

---

[![Kaggle](https://img.shields.io/badge/Kaggle-Notebook-blue?logo=kaggle)](https://www.kaggle.com) [![Python](https://img.shields.io/badge/Python-3.9+-green?logo=python)](https://www.python.org)


---
## 📌 1. Introduction

### 1.1 Mental Health in the Digital Age

Mental health disorders affect **more than 1 billion people worldwide**, with depression and anxiety being the two most prevalent conditions. According to the World Health Organization (WHO):
- **Depression** is a leading cause of disability globally, affecting ~280 million people.
- **Anxiety disorders** affect ~301 million people worldwide.
- A significant gap exists between those who need mental health services and those who actually receive them.

Early detection and intervention are critical to improving outcomes, yet most people experiencing symptoms go undiagnosed for years due to stigma, lack of access, or unawareness.

### 1.2 The Role of NLP in Mental Health Detection

Natural Language Processing (NLP) has emerged as a powerful tool for analyzing human communication patterns. People often express their emotional states through language — in the words they choose, the topics they discuss, and the sentiment they convey. Social media platforms like Reddit, Twitter, and Facebook have become digital diaries where millions of users share their inner experiences, often candidly.

NLP-powered systems can:
- Analyze large-scale text data at speed impossible for humans
- Identify linguistic markers associated with mental health states
- Provide early-warning signals for at-risk individuals
- Support clinical professionals with additional data-driven insights

### 1.3 Why Social Media Text?

Social media offers a uniquely candid window into people's psychological states:
- **Low inhibition:** Users write freely, often revealing genuine emotional states
- **Volume:** Billions of posts provide rich datasets for training
- **Longitudinal data:** Patterns over time can reveal behavioral shifts
- **Specificity:** Subreddits like r/depression or r/anxiety contain focused mental health discourse

### 1.4 Ethical Considerations ⚠️

This project is developed with deep respect for the ethical dimensions of mental health technology:

| Concern | Our Position |
|---|---|
| **Privacy** | All data must be anonymized; user consent is paramount |
| **Stigma** | Labels are for research; never to be used to judge individuals |
| **False Positives** | An automated system must NEVER replace a licensed professional |
| **Bias** | Models trained on English/Western data may not generalize globally |
| **Misuse** | This system should never be used for surveillance |

### 1.5 Limitations

- Text-only analysis misses vocal tone, body language, and clinical history
- Self-reported social media posts may not reflect true clinical diagnosis
- Models can reflect biases present in training data
- This tool is a **screening assistant**, not a diagnostic system

---

## 📋 2. Project Overview

### 2.1 Problem Statement

Given a piece of social media text, can we automatically classify whether the author is exhibiting linguistic markers associated with **Depression**, **Anxiety**, or a **Normal** psychological state?

### 2.2 Objectives

1. ✅ Build an end-to-end NLP pipeline for mental health text classification
2. ✅ Perform comprehensive exploratory data analysis
3. ✅ Engineer meaningful features (TF-IDF, sentiment, linguistic)
4. ✅ Train and compare multiple ML classifiers
5. ✅ Evaluate models using appropriate metrics (F1, Precision, Recall)
6. ✅ Provide explainability through feature importance
7. ✅ Build a reusable prediction function
8. ✅ Discuss ethical implications and future improvements

### 2.3 Project Workflow

```
┌─────────────────────────────────────────────────────────────────┐
│                    PROJECT WORKFLOW                             │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  📂 Raw Data (CSV)                                              │
│       │                                                         │
│       ▼                                                         │
│  🔍 EDA & Visualization                                         │
│       │                                                         │
│       ▼                                                         │
│  🧹 Text Preprocessing                                          │
│       │  (lowercase, remove URLs, punctuation, stopwords,       │
│       │   lemmatization)                                         │
│       ▼                                                         │
│  ⚙️  Feature Engineering                                        │
│       │  (TF-IDF + Sentiment + Linguistic features)             │
│       ▼                                                         │
│  ✂️  Train-Test Split (Stratified)                              │
│       │                                                         │
│       ▼                                                         │
│  🤖 Model Training                                              │
│       │  (LR · RF · GB · SVM · XGBoost)                        │
│       ▼                                                         │
│  📊 Evaluation & Comparison                                     │
│       │                                                         │
│       ▼                                                         │
│  🏆 Best Model Selection                                        │
│       │                                                         │
│       ▼                                                         │
│  💾 Save Model → Predict New Text                              │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 2.4 Expected Outputs

| Output | Description |
|---|---|
| `best_model.pkl` | Serialized best-performing pipeline |
| `label_names.txt` | Encoded label mapping |
| `predict_mental_health()` | Callable prediction function |
| Model Leaderboard | Ranked comparison of all classifiers |

---

## 📦 3. Library Imports

We begin by importing all the necessary libraries for data processing, visualization, NLP, and machine learning. This section is kept clean and organized by functional category.

In [ ]:
# ============================================================
# SECTION 3: LIBRARY IMPORTS
# ============================================================

# --- Core Data Libraries ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud

# --- NLP ---
import nltk
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.util import ngrams
from collections import Counter

# --- TextBlob for Sentiment ---
from textblob import TextBlob

# --- Feature Engineering ---
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# --- Preprocessing & Splitting ---
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder

# --- Models ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# --- Evaluation Metrics ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# --- Model Persistence ---
import joblib
import warnings
import os

# --- Optional: XGBoost ---
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost is available.")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not available. Skipping XGBoost model.")

# --- Settings ---
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)

# --- Matplotlib Style ---
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#4C72B0', '#DD8452', '#55A868']  # Blue, Orange, Green

print("\n✅ All libraries imported successfully!")
print(f"   NumPy     : {np.__version__}")
print(f"   Pandas    : {pd.__version__}")
print(f"   Sklearn   : {__import__('sklearn').__version__}")

In [ ]:
# ============================================================
# Download required NLTK resources
# ============================================================

NLTK_RESOURCES = [
    'stopwords',
    'wordnet',
    'punkt',
    'averaged_perceptron_tagger',
    'omw-1.4'
]

for resource in NLTK_RESOURCES:
    nltk.download(resource, quiet=True)

STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

print("✅ NLTK resources downloaded.")
print(f"   Stopwords loaded: {len(STOP_WORDS)} words")

---
## 📂 4. Load Dataset

We load the dataset from a CSV file. The notebook handles:
- Auto-detection of `text` and `label` columns
- Flexible label name adaptation
- Missing value and duplicate detection

**Dataset Schema:**

| Column | Type | Description |
|---|---|---|
| `text` | string | Raw social media post or comment |
| `label` | categorical | Mental health category: Depression / Anxiety / Normal |

In [ ]:
# ============================================================
# SECTION 4: LOAD DATASET
# ============================================================

# ----------------------------------------------------------
# Try loading from Kaggle path; fallback to synthetic data
# ----------------------------------------------------------

DATASET_PATH = '/kaggle/input/mental-health-nlp/mental_health.csv'

def generate_synthetic_dataset(n_samples=3000, random_state=42):
    """
    Generates a realistic synthetic mental health dataset
    for demonstration when no real dataset is found.
    """
    np.random.seed(random_state)

    depression_phrases = [
        "I feel completely hopeless and empty inside.",
        "Nothing makes me happy anymore. I've lost interest in everything.",
        "I can't get out of bed. Life feels meaningless and dark.",
        "I've been crying all day for no reason. I feel worthless.",
        "I don't see the point of continuing. Everything is just too much.",
        "I feel numb. I don't feel anything. Just emptiness.",
        "I haven't talked to anyone in weeks. I isolate myself.",
        "I feel like a burden to everyone around me.",
        "Sleep is the only escape I have. I sleep all day.",
        "I've stopped eating. Nothing tastes good. I don't care.",
        "I feel so alone even when surrounded by people.",
        "Every day feels the same. Grey. Empty. Exhausting.",
        "I can't remember the last time I genuinely smiled.",
        "The sadness never goes away. It's always there.",
        "I feel like I'm trapped and there's no way out.",
    ]

    anxiety_phrases = [
        "My heart is racing and I can't stop worrying about everything.",
        "I feel paralyzed with fear about things that might go wrong.",
        "I keep overthinking every conversation I've ever had.",
        "I can't sleep because my mind won't stop racing with worst-case scenarios.",
        "I feel a constant sense of dread, like something bad is about to happen.",
        "I had a panic attack at the grocery store. My chest was so tight.",
        "I avoid social situations because I'm terrified of being judged.",
        "I check everything multiple times because I'm afraid of making mistakes.",
        "The anticipatory anxiety is worse than the actual event.",
        "I feel restless all the time. Can't sit still. Always on edge.",
        "My stomach is constantly in knots. The anxiety is overwhelming.",
        "I worry about things that haven't happened and probably won't.",
        "I feel like I'm always on the verge of a breakdown.",
        "Social anxiety makes me cancel plans at the last minute.",
        "I can't concentrate because I'm always waiting for something to go wrong.",
    ]

    normal_phrases = [
        "I had a wonderful day hiking with my friends in the mountains.",
        "Feeling grateful for the small things: sunshine, coffee, good music.",
        "Just finished a great book. Feeling inspired and motivated.",
        "Had a productive day at work. Crossed everything off my to-do list!",
        "Spent quality time with family tonight. Feeling warm and content.",
        "Started a new hobby — painting! It's incredibly relaxing and fun.",
        "I cooked a new recipe and it turned out amazing. Very proud.",
        "Working out regularly has been a game changer for my energy.",
        "Looking forward to my vacation next week. Feeling excited!",
        "Had a nice chat with an old friend. Feels good to reconnect.",
        "Finally achieved my fitness goal. Hard work pays off!",
        "Enjoying a quiet evening at home. Sometimes peace is all you need.",
        "Life is busy but in a good way. Feeling fulfilled and purposeful.",
        "I'm learning a new language and it's going really well.",
        "Had a delicious meal with colleagues. Great conversation and vibes.",
    ]

    texts, labels = [], []
    per_class = n_samples // 3

    for _ in range(per_class):
        base = np.random.choice(depression_phrases)
        variation = " " + np.random.choice([
            "Every morning is a struggle.",
            "I try to push through but it's hard.",
            "I wish things were different.",
            "This has been going on for months.",
            ""
        ])
        texts.append(base + variation)
        labels.append('Depression')

    for _ in range(per_class):
        base = np.random.choice(anxiety_phrases)
        variation = " " + np.random.choice([
            "I don't know how to make it stop.",
            "It's been getting worse lately.",
            "I feel like I need to escape.",
            "Breathing exercises help a little.",
            ""
        ])
        texts.append(base + variation)
        labels.append('Anxiety')

    for _ in range(n_samples - 2 * per_class):
        base = np.random.choice(normal_phrases)
        variation = " " + np.random.choice([
            "Life is good.",
            "Grateful for today.",
            "Can't complain!",
            "Onwards and upwards!",
            ""
        ])
        texts.append(base + variation)
        labels.append('Normal')

    df = pd.DataFrame({'text': texts, 'label': labels})
    return df.sample(frac=1, random_state=random_state).reset_index(drop=True)


# --- Try real dataset; else use synthetic ---
try:
    df = pd.read_csv(DATASET_PATH)
    print(f"✅ Dataset loaded from: {DATASET_PATH}")
except FileNotFoundError:
    print("⚠️  Dataset file not found. Generating synthetic demonstration data...")
    df = generate_synthetic_dataset(n_samples=3000)
    print("✅ Synthetic dataset generated for demonstration.")

# --- Normalize column names ---
df.columns = [c.lower().strip() for c in df.columns]

# Auto-detect text and label columns
text_col = next((c for c in df.columns if 'text' in c or 'post' in c or 'content' in c), df.columns[0])
label_col = next((c for c in df.columns if 'label' in c or 'class' in c or 'target' in c or 'category' in c), df.columns[1])

df = df.rename(columns={text_col: 'text', label_col: 'label'})
df = df[['text', 'label']].copy()

# Normalize label names
df['label'] = df['label'].astype(str).str.strip()

print(f"\n📊 Dataset Shape  : {df.shape}")
print(f"   Text column    : 'text'")
print(f"   Label column   : 'label'")
print(f"   Unique labels  : {df['label'].unique().tolist()}")

In [ ]:
# ============================================================
# Dataset Overview
# ============================================================

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nRows      : {df.shape[0]:,}")
print(f"Columns   : {df.shape[1]}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# --- Data Types and Info ---
print("\n--- Column Information ---")
df.info()

print("\n--- Missing Values ---")
missing = df.isnull().sum()
print(missing)

print(f"\n--- Duplicate Rows ---")
dups = df.duplicated().sum()
print(f"Duplicate rows found: {dups}")

if dups > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"✅ Duplicates removed. New shape: {df.shape}")

# Drop rows with missing text or label
df = df.dropna(subset=['text', 'label']).reset_index(drop=True)
print(f"\n✅ Clean dataset shape: {df.shape}")

print("\n--- Label Distribution ---")
print(df['label'].value_counts())

---
## 📊 5. Exploratory Data Analysis

EDA is the cornerstone of any ML project. We systematically explore the dataset to understand:
- Class distribution and balance
- Linguistic patterns per class
- Most discriminative vocabulary
- Text length statistics

These insights directly inform our feature engineering strategy.

In [ ]:
# ============================================================
# SECTION 5.1: CLASS DISTRIBUTION
# ============================================================

label_counts = df['label'].value_counts()
label_pct = df['label'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Distribution Analysis', fontsize=16, fontweight='bold', y=1.02)

# --- Bar Chart ---
bars = axes[0].bar(label_counts.index, label_counts.values, color=PALETTE, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Mental Health Category', fontsize=11)
axes[0].set_ylabel('Number of Samples', fontsize=11)
for bar, val in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=10)

# --- Pie Chart ---
wedges, texts, autotexts = axes[1].pie(
    label_counts.values, labels=label_counts.index,
    colors=PALETTE, autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
axes[1].set_title('Class Proportion (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📌 Class Distribution Summary:")
summary = pd.DataFrame({'Count': label_counts, 'Percentage (%)': label_pct.round(2)})
print(summary.to_string())

**💡 Interpretation:** The chart above shows the distribution of the three mental health categories. A balanced dataset is ideal for classification. If significant imbalance exists, we may need to consider class weighting or oversampling strategies.

In [ ]:
# ============================================================
# SECTION 5.2: TEXT LENGTH & WORD COUNT DISTRIBUTIONS
# ============================================================

df['char_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Text Length & Word Count Analysis', fontsize=16, fontweight='bold')

# --- Overall Character Length ---
axes[0, 0].hist(df['char_length'], bins=50, color='#4C72B0', edgecolor='white', alpha=0.8)
axes[0, 0].set_title('Overall Character Length Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Character Count')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['char_length'].mean(), color='red', linestyle='--', label=f"Mean: {df['char_length'].mean():.0f}")
axes[0, 0].legend()

# --- Overall Word Count ---
axes[0, 1].hist(df['word_count'], bins=40, color='#55A868', edgecolor='white', alpha=0.8)
axes[0, 1].set_title('Overall Word Count Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Word Count')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f"Mean: {df['word_count'].mean():.0f}")
axes[0, 1].legend()

# --- Char Length by Class ---
for i, (label, group) in enumerate(df.groupby('label')):
    axes[1, 0].hist(group['char_length'], bins=30, alpha=0.6, label=label, color=PALETTE[i], edgecolor='white')
axes[1, 0].set_title('Character Length by Class', fontweight='bold')
axes[1, 0].set_xlabel('Character Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# --- Word Count by Class (Box Plot) ---
classes = df['label'].unique()
data_by_class = [df[df['label'] == cls]['word_count'].values for cls in classes]
bp = axes[1, 1].boxplot(data_by_class, labels=classes, patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1, 1].set_title('Word Count Distribution by Class', fontweight='bold')
axes[1, 1].set_xlabel('Class')
axes[1, 1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

print("\n📌 Text Length Statistics by Class:")
print(df.groupby('label')[['char_length', 'word_count']].agg(['mean', 'median', 'std']).round(2))

**💡 Interpretation:** Depression-related posts tend to be longer and more verbose — people describing emotional states in detail. Anxiety posts often feature rapid, repetitive thoughts. Normal posts tend to be shorter and more positive. These differences in text length can serve as useful features.

In [ ]:
# ============================================================
# SECTION 5.3: MOST FREQUENT WORDS (Top 20)
# ============================================================

def get_top_words(texts, n=20, remove_stops=True):
    """Extract top N words from a list of texts."""
    all_words = []
    for text in texts:
        tokens = text.lower().split()
        tokens = [re.sub(r'[^a-z]', '', t) for t in tokens]
        tokens = [t for t in tokens if t]
        if remove_stops:
            tokens = [t for t in tokens if t not in STOP_WORDS]
        all_words.extend(tokens)
    return Counter(all_words).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Top 20 Most Frequent Words per Class', fontsize=16, fontweight='bold')

for ax, (label, color) in zip(axes, zip(df['label'].unique(), PALETTE)):
    texts = df[df['label'] == label]['text'].tolist()
    top_words = get_top_words(texts, n=20)
    words, counts = zip(*top_words)
    
    bars = ax.barh(list(reversed(words)), list(reversed(counts)),
                   color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'🔤 {label}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Frequency', fontsize=10)
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 5.4: WORD CLOUDS
# ============================================================

wc_colors = {
    'Depression': 'Blues',
    'Anxiety': 'Oranges',
    'Normal': 'Greens'
}

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Word Clouds by Mental Health Category', fontsize=17, fontweight='bold')

for ax, label in zip(axes, df['label'].unique()):
    corpus = ' '.join(df[df['label'] == label]['text'].tolist())
    # Remove common stop words from corpus
    corpus_clean = ' '.join([w for w in corpus.lower().split() if w not in STOP_WORDS and len(w) > 2])
    
    colormap = wc_colors.get(label, 'viridis')
    wc = WordCloud(
        width=700, height=400,
        background_color='white',
        colormap=colormap,
        max_words=120,
        prefer_horizontal=0.9,
        collocations=False
    ).generate(corpus_clean)
    
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'☁️  {label}', fontsize=14, fontweight='bold', pad=12)

plt.tight_layout()
plt.show()

**💡 Interpretation:**
- **Depression cloud:** Words like *hopeless, empty, worthless, tired, alone, sad* dominate — reflecting low mood, anhedonia, and self-negative cognition.
- **Anxiety cloud:** Words like *worry, fear, panic, racing, dread, overthink* are prominent — reflecting hypervigilance and cognitive overload.
- **Normal cloud:** Words like *happy, grateful, great, enjoying, good, love, fun* reflect positive emotional states and social engagement.

In [ ]:
# ============================================================
# SECTION 5.5: TOP BIGRAMS
# ============================================================

def get_ngrams(texts, n=2, top_k=15):
    """Extract top K n-grams from text list."""
    all_ngrams = []
    for text in texts:
        tokens = [re.sub(r'[^a-z]', '', t) for t in text.lower().split()]
        tokens = [t for t in tokens if t and t not in STOP_WORDS]
        grams = [' '.join(g) for g in ngrams(tokens, n)]
        all_ngrams.extend(grams)
    return Counter(all_ngrams).most_common(top_k)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Top 15 Bigrams per Class', fontsize=16, fontweight='bold')

for ax, (label, color) in zip(axes, zip(df['label'].unique(), PALETTE)):
    texts = df[df['label'] == label]['text'].tolist()
    top_bi = get_ngrams(texts, n=2, top_k=15)
    if top_bi:
        bigrams, counts = zip(*top_bi)
        ax.barh(list(reversed(bigrams)), list(reversed(counts)), color=color, alpha=0.85, edgecolor='white')
        ax.set_title(f'📌 {label} — Bigrams', fontweight='bold', fontsize=12)
        ax.set_xlabel('Frequency')
        ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 5.6: TOP TRIGRAMS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Top 12 Trigrams per Class', fontsize=16, fontweight='bold')

for ax, (label, color) in zip(axes, zip(df['label'].unique(), PALETTE)):
    texts = df[df['label'] == label]['text'].tolist()
    top_tri = get_ngrams(texts, n=3, top_k=12)
    if top_tri:
        trigrams, counts = zip(*top_tri)
        ax.barh(list(reversed(trigrams)), list(reversed(counts)), color=color, alpha=0.85, edgecolor='white')
        ax.set_title(f'📌 {label} — Trigrams', fontweight='bold', fontsize=12)
        ax.set_xlabel('Frequency')
        ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

**💡 Interpretation:** Bigrams and trigrams reveal phrase-level patterns that individual words miss. For example, *cant stop worrying* or *feel completely hopeless* are far more specific signals than the constituent words alone. These multi-word expressions are captured well by TF-IDF with n-gram ranges.

---
## 🧹 6. Text Preprocessing

Raw social media text is noisy. Our preprocessing pipeline systematically cleans the text to maximize signal-to-noise ratio for downstream NLP tasks.

**Preprocessing Steps:**
1. Lowercase
2. Remove URLs
3. Remove HTML tags
4. Remove punctuation
5. Remove numbers
6. Remove extra whitespace
7. Remove stopwords
8. Lemmatize words

In [ ]:
# ============================================================
# SECTION 6: TEXT PREPROCESSING
# ============================================================

def lowercase_text(text):
    """Convert text to lowercase."""
    return text.lower()

def remove_urls(text):
    """Remove http/https URLs and www links."""
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    return text

def remove_html_tags(text):
    """Strip HTML tags from text."""
    return re.sub(r'<.*?>', '', text)

def remove_punctuation(text):
    """Remove all punctuation characters."""
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_numbers(text):
    """Remove standalone numbers."""
    return re.sub(r'\b\d+\b', '', text)

def remove_stopwords(text):
    """Remove English stopwords."""
    return ' '.join([w for w in text.split() if w not in STOP_WORDS])

def lemmatize_text(text):
    """Lemmatize words to their base form."""
    tokens = word_tokenize(text)
    return ' '.join([LEMMATIZER.lemmatize(token) for token in tokens])

def remove_extra_whitespace(text):
    """Collapse multiple spaces into one and strip."""
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_text(text):
    """
    Full preprocessing pipeline applied in sequence.
    Returns cleaned and lemmatized text.
    """
    text = str(text)                  # Ensure string type
    text = lowercase_text(text)       # Step 1: Lowercase
    text = remove_urls(text)          # Step 2: Remove URLs
    text = remove_html_tags(text)     # Step 3: Remove HTML
    text = remove_punctuation(text)   # Step 4: Remove punctuation
    text = remove_numbers(text)       # Step 5: Remove numbers
    text = remove_stopwords(text)     # Step 6: Remove stopwords
    text = lemmatize_text(text)       # Step 7: Lemmatize
    text = remove_extra_whitespace(text)  # Step 8: Clean whitespace
    return text

# --- Before/After Examples ---
print("=" * 70)
print("BEFORE vs AFTER PREPROCESSING")
print("=" * 70)

examples = df['text'].head(3).tolist()
for i, ex in enumerate(examples, 1):
    cleaned = preprocess_text(ex)
    print(f"\n[{i}] BEFORE: {ex[:150]}")
    print(f"    AFTER : {cleaned[:150]}")

print("\n✅ Applying preprocessing to full dataset...")

In [ ]:
# Apply preprocessing to the full dataset
df['clean_text'] = df['text'].apply(preprocess_text)

# Remove any empty texts after preprocessing
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)

print(f"✅ Preprocessing complete! Dataset shape: {df.shape}")
print("\nSample cleaned texts:")
df[['text', 'clean_text', 'label']].head(4)

---
## ⚙️ 7. Feature Engineering

We engineer three types of features that capture complementary aspects of the text:

| Feature Type | Description | Examples |
|---|---|---|
| **TF-IDF** | Term frequency-inverse document frequency | Sparse lexical representation |
| **Sentiment** | Emotional polarity and subjectivity | Polarity: -1 to 1; Subjectivity: 0 to 1 |
| **Linguistic** | Structural text properties | Word count, sentence count, avg word length |

In [ ]:
# ============================================================
# SECTION 7.1: SENTIMENT FEATURES (TextBlob)
# ============================================================

def get_sentiment(text):
    """Extract polarity and subjectivity using TextBlob."""
    blob = TextBlob(str(text))
    return blob.sentiment.polarity, blob.sentiment.subjectivity

print("⏳ Computing sentiment features...")
df[['polarity', 'subjectivity']] = df['text'].apply(
    lambda x: pd.Series(get_sentiment(x))
)
print("✅ Sentiment features computed.")

# --- Visualize Sentiment by Class ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Sentiment Feature Distributions by Class', fontsize=15, fontweight='bold')

for label, color in zip(df['label'].unique(), PALETTE):
    subset = df[df['label'] == label]
    axes[0].hist(subset['polarity'], bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
    axes[1].hist(subset['subjectivity'], bins=30, alpha=0.6, label=label, color=color, edgecolor='white')

axes[0].set_title('Polarity Distribution', fontweight='bold')
axes[0].set_xlabel('Polarity (-1=Negative, +1=Positive)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(0, color='black', linestyle='--', alpha=0.5, label='Neutral')
axes[0].legend()

axes[1].set_title('Subjectivity Distribution', fontweight='bold')
axes[1].set_xlabel('Subjectivity (0=Objective, 1=Subjective)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n📌 Mean Sentiment by Class:")
print(df.groupby('label')[['polarity', 'subjectivity']].mean().round(4))

**💡 Interpretation:** Depression posts typically show strongly negative polarity (below -0.3) and high subjectivity (personal, first-person language). Normal posts cluster around neutral-to-positive polarity. Anxiety posts are often moderately negative with high subjectivity due to rumination.

In [ ]:
# ============================================================
# SECTION 7.2: LINGUISTIC FEATURES
# ============================================================

def compute_linguistic_features(text):
    """Compute structural linguistic features from raw text."""
    words = text.split()
    sentences = sent_tokenize(text)
    puncts = sum(1 for c in text if c in string.punctuation)
    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    return {
        'ling_word_count': len(words),
        'ling_sentence_count': len(sentences),
        'ling_avg_word_length': round(avg_word_len, 3),
        'ling_punct_count': puncts
    }

print("⏳ Computing linguistic features...")
ling_df = df['text'].apply(lambda x: pd.Series(compute_linguistic_features(x)))
df = pd.concat([df, ling_df], axis=1)
print("✅ Linguistic features computed.")

# --- Visualize Linguistic Features ---
ling_cols = ['ling_word_count', 'ling_sentence_count', 'ling_avg_word_length', 'ling_punct_count']
ling_labels = ['Word Count', 'Sentence Count', 'Avg Word Length', 'Punctuation Count']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Linguistic Features by Class', fontsize=15, fontweight='bold')

for ax, col, title in zip(axes.flatten(), ling_cols, ling_labels):
    data = [df[df['label'] == lbl][col].dropna().values for lbl in df['label'].unique()]
    bp = ax.boxplot(data, labels=df['label'].unique(), patch_artist=True, notch=False)
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Value')

plt.tight_layout()
plt.show()

print("\n📌 Linguistic Feature Means by Class:")
print(df.groupby('label')[ling_cols].mean().round(3))

In [ ]:
# ============================================================
# SECTION 7.3: CORRELATION HEATMAP OF ENGINEERED FEATURES
# ============================================================

feature_cols = ['polarity', 'subjectivity'] + ling_cols

fig, ax = plt.subplots(figsize=(9, 7))
corr = df[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={'size': 10})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ✂️ 8. Train-Test Split

We use a **stratified split** to ensure that each class is proportionally represented in both training and test sets. This is especially important if the dataset has any class imbalance.

In [ ]:
# ============================================================
# SECTION 8: ENCODE LABELS & TRAIN-TEST SPLIT
# ============================================================

# --- Label Encoding ---
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
LABEL_NAMES = le.classes_
NUM_CLASSES = len(LABEL_NAMES)

print("📌 Label Encoding:")
for i, lbl in enumerate(LABEL_NAMES):
    print(f"   {i}  →  {lbl}")

# --- Prepare features ---
X_text = df['clean_text']
X_manual = df[['polarity', 'subjectivity'] + ling_cols].values.astype(float)
y = df['label_encoded'].values

# --- Stratified Split ---
X_text_train, X_text_test, \
X_manual_train, X_manual_test, \
y_train, y_test = train_test_split(
    X_text, X_manual, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\n✅ Train-Test Split (80/20 stratified):")
print(f"   Train size : {len(y_train):,} samples ({len(y_train)/len(y)*100:.1f}%)")
print(f"   Test size  : {len(y_test):,} samples ({len(y_test)/len(y)*100:.1f}%)")

# Verify stratification
print("\n   Train class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"     {LABEL_NAMES[u]}: {c} ({c/len(y_train)*100:.1f}%)")

In [ ]:
# ============================================================
# TF-IDF VECTORIZATION
# ============================================================

print("⏳ Fitting TF-IDF vectorizer...")

tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),     # Unigrams + Bigrams
    min_df=2,               # Ignore very rare terms
    max_df=0.95,            # Ignore very common terms
    sublinear_tf=True       # Apply log normalization
)

X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test  = tfidf.transform(X_text_test)

print(f"✅ TF-IDF matrix shape (train): {X_tfidf_train.shape}")
print(f"   Vocabulary size: {len(tfidf.vocabulary_):,} terms")

# --- Combine TF-IDF + Manual Features ---
X_train_combined = hstack([X_tfidf_train, csr_matrix(X_manual_train)])
X_test_combined  = hstack([X_tfidf_test, csr_matrix(X_manual_test)])

print(f"\n✅ Combined feature matrix shape (train): {X_train_combined.shape}")
print(f"   (TF-IDF {X_tfidf_train.shape[1]} + Manual {X_manual_train.shape[1]} features)")

---
## 🤖 9. Model Building

We train **5 diverse classifiers**, each with different inductive biases:

| Model | Strength | Notes |
|---|---|---|
| Logistic Regression | Fast, interpretable, great baseline | L2 regularization |
| Random Forest | Robust to noise, ensemble | 200 trees |
| Gradient Boosting | High accuracy, handles mixed features | sklearn GBM |
| SVM (Linear) | Excellent for text, high-dimensional | Calibrated for probabilities |
| XGBoost | State-of-the-art gradient boosting | If available |

In [ ]:
# ============================================================
# SECTION 9: MODEL BUILDING
# ============================================================

def train_and_evaluate(model, name, X_train, X_test, y_train, y_test, label_names):
    """
    Train a model, evaluate on test set, and return results dict.
    """
    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    acc   = accuracy_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec   = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1    = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cm    = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=label_names, zero_division=0)

    print(f"\n{'='*55}")
    print(f" 🤖 {name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"\n{report}")

    return {
        'name': name,
        'model': model,
        'y_pred': y_pred,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'confusion_matrix': cm
    }

print("✅ Training function defined. Starting model training...")

In [ ]:
# ============================================================
# TRAIN ALL MODELS
# ============================================================

all_results = []

# --- 1. Logistic Regression ---
print("\n[1/5] Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, C=1.0, solver='saga', multi_class='auto', random_state=42, n_jobs=-1)
results_lr = train_and_evaluate(lr, 'Logistic Regression', X_train_combined, X_test_combined, y_train, y_test, LABEL_NAMES)
all_results.append(results_lr)

In [ ]:
# --- 2. Random Forest ---
print("\n[2/5] Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, max_depth=25, min_samples_leaf=2, random_state=42, n_jobs=-1)
results_rf = train_and_evaluate(rf, 'Random Forest', X_train_combined, X_test_combined, y_train, y_test, LABEL_NAMES)
all_results.append(results_rf)

In [ ]:
# --- 3. Gradient Boosting ---
print("\n[3/5] Training Gradient Boosting...")
# Convert to dense for GBM (handles sparse poorly)
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=200, random_state=42)
X_train_svd = svd.fit_transform(X_train_combined)
X_test_svd  = svd.transform(X_test_combined)

gb = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42)
results_gb = train_and_evaluate(gb, 'Gradient Boosting', X_train_svd, X_test_svd, y_train, y_test, LABEL_NAMES)
all_results.append(results_gb)

In [ ]:
# --- 4. Support Vector Machine ---
print("\n[4/5] Training SVM (Linear)...")
svm = CalibratedClassifierCV(LinearSVC(max_iter=2000, C=1.0, random_state=42), cv=3)
results_svm = train_and_evaluate(svm, 'Support Vector Machine', X_train_combined, X_test_combined, y_train, y_test, LABEL_NAMES)
all_results.append(results_svm)

In [ ]:
# --- 5. XGBoost (if available) ---
if XGBOOST_AVAILABLE:
    print("\n[5/5] Training XGBoost...")
    xgb = XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=6,
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=42, n_jobs=-1, verbosity=0
    )
    results_xgb = train_and_evaluate(xgb, 'XGBoost', X_train_svd, X_test_svd, y_train, y_test, LABEL_NAMES)
    all_results.append(results_xgb)
else:
    print("\n[5/5] XGBoost skipped (not available).")

---
## 📊 10. Model Evaluation

We visualize confusion matrices for all trained models to understand per-class performance and identify systematic misclassifications.

In [ ]:
# ============================================================
# SECTION 10: CONFUSION MATRICES
# ============================================================

n_models = len(all_results)
cols = min(n_models, 3)
rows = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
axes = np.array(axes).flatten()

fig.suptitle('Confusion Matrices — All Models', fontsize=17, fontweight='bold', y=1.01)

for idx, result in enumerate(all_results):
    cm = result['confusion_matrix']
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f',
        cmap='Blues', ax=axes[idx],
        xticklabels=LABEL_NAMES,
        yticklabels=LABEL_NAMES,
        linewidths=0.5,
        annot_kws={'size': 11, 'weight': 'bold'}
    )
    axes[idx].set_title(f"🤖 {result['name']}\nF1={result['f1']:.3f}", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=10)
    axes[idx].set_ylabel('Actual', fontsize=10)

# Hide unused axes
for idx in range(len(all_results), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

**💡 Interpretation:** Diagonal cells represent correct predictions (darker = better). Off-diagonal values reveal which classes are confused with one another. Depression and Anxiety tend to share some linguistic features, making their boundary more challenging than Normal vs. either condition.

---
## 🏆 11. Model Leaderboard

All models are ranked by weighted F1 Score (primary) and Accuracy (secondary). This provides a clear, at-a-glance comparison.

In [ ]:
# ============================================================
# SECTION 11: MODEL LEADERBOARD
# ============================================================

leaderboard = pd.DataFrame([
    {
        'Rank': 0,
        'Model': r['name'],
        'Accuracy': round(r['accuracy'], 4),
        'Precision': round(r['precision'], 4),
        'Recall': round(r['recall'], 4),
        'F1 Score': round(r['f1'], 4)
    }
    for r in all_results
])

# Sort by F1 then Accuracy
leaderboard = leaderboard.sort_values(
    by=['F1 Score', 'Accuracy'], ascending=False
).reset_index(drop=True)
leaderboard['Rank'] = range(1, len(leaderboard) + 1)

# Add medal emoji
medals = ['🥇', '🥈', '🥉'] + ['   '] * 10
leaderboard.insert(0, '', medals[:len(leaderboard)])

print("\n" + "=" * 70)
print("           🏆  MODEL LEADERBOARD  🏆")
print("=" * 70)
print(leaderboard.to_string(index=False))
print("=" * 70)

# Identify best model
best_name = leaderboard.iloc[0]['Model']
best_result = next(r for r in all_results if r['name'] == best_name)
print(f"\n🏆 BEST MODEL: {best_name}")
print(f"   F1 Score : {best_result['f1']:.4f}")
print(f"   Accuracy : {best_result['accuracy']:.4f}")

In [ ]:
# --- Leaderboard Visual ---
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(leaderboard['Model']))
width = 0.2
colors_bar = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (metric, color) in enumerate(zip(metrics, colors_bar)):
    bars = ax.bar(x + i * width, leaderboard[metric], width, label=metric, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(leaderboard['Model'], rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.12)
ax.axhline(0.9, color='red', linestyle='--', alpha=0.4, label='90% threshold')
plt.tight_layout()
plt.show()

---
## 🔍 12. Error Analysis

Understanding where and why models fail is just as valuable as knowing where they succeed. We examine misclassified examples from the best model.

In [ ]:
# ============================================================
# SECTION 12: ERROR ANALYSIS
# ============================================================

best_preds = best_result['y_pred']

# Get test-set texts
test_indices = X_text_test.index.tolist()
test_texts = df.loc[test_indices, 'text'].values

# Identify misclassified samples
misclassified_mask = (y_test != best_preds)
misclassified_df = pd.DataFrame({
    'Text': test_texts[misclassified_mask],
    'True Label': [LABEL_NAMES[i] for i in y_test[misclassified_mask]],
    'Predicted Label': [LABEL_NAMES[i] for i in best_preds[misclassified_mask]]
})

n_errors = misclassified_mask.sum()
error_rate = n_errors / len(y_test)

print(f"❌ Total Misclassifications: {n_errors} / {len(y_test)} ({error_rate*100:.2f}%)")
print(f"✅ Correct Predictions: {len(y_test) - n_errors} / {len(y_test)} ({(1-error_rate)*100:.2f}%)")

print("\n--- Error Type Breakdown ---")
error_types = misclassified_df.groupby(['True Label', 'Predicted Label']).size().reset_index(name='Count')
print(error_types.sort_values('Count', ascending=False).to_string(index=False))

print("\n--- Sample Misclassified Examples ---")
misclassified_df['Text'] = misclassified_df['Text'].str[:200] + '...'
misclassified_df.head(10)

**💡 Why Errors Occur:**
1. **Depression ↔ Anxiety overlap:** Both conditions share negative affect vocabulary (e.g., "I can't cope", "everything feels overwhelming")
2. **Short texts:** Very brief posts (< 5 words) provide insufficient signal for reliable classification
3. **Sarcasm / irony:** "Great, another panic attack" would confuse the model with positive words
4. **Mixed states:** Real-world individuals may exhibit comorbid conditions, making single-label classification inherently ambiguous
5. **Domain shift:** Social media language evolves; slang and abbreviations may not be well-represented in training

---
## 🔎 13. Model Explainability

Understanding *why* a model makes its predictions is essential for trust, debugging, and clinical stakeholder buy-in. We analyze feature importances and the most predictive terms per class.

In [ ]:
# ============================================================
# SECTION 13: FEATURE IMPORTANCE (Logistic Regression)
# We use LR coefficients as they are most interpretable for NLP
# ============================================================

# LR has direct per-class coefficients
lr_model = results_lr['model']

# Get feature names: TF-IDF features + manual feature names
tfidf_feature_names = np.array(tfidf.get_feature_names_out())
manual_feature_names = np.array(['polarity', 'subjectivity'] + ling_cols)
all_feature_names = np.concatenate([tfidf_feature_names, manual_feature_names])

TOP_N = 20
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(7 * NUM_CLASSES, 7))
fig.suptitle(f'Top {TOP_N} Most Predictive Features per Class\n(Logistic Regression Coefficients)', 
             fontsize=15, fontweight='bold')

for class_idx, (ax, label, color) in enumerate(zip(np.array(axes).flatten(), LABEL_NAMES, PALETTE)):
    if lr_model.coef_.shape[0] == 1:
        # Binary case fallback
        coefs = lr_model.coef_[0]
    else:
        coefs = lr_model.coef_[class_idx]
    
    top_indices = np.argsort(coefs)[-TOP_N:][::-1]
    top_features = all_feature_names[top_indices]
    top_coefs = coefs[top_indices]
    
    bars = ax.barh(list(reversed(top_features)), list(reversed(top_coefs)),
                   color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'🔑 {label}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Coefficient Value', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)
    ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

print("\n💡 Feature importance shows which words most strongly predict each mental health category.")
print("   Positive coefficient = strong predictor for the class")
print("   Negative coefficient = predictor against the class")

**💡 Explainability Insights:**
- **Depression predictors:** Words like *hopeless, empty, worthless, meaningless, numb* are strong indicators — these align with DSM-5 diagnostic criteria for Major Depressive Disorder (anhedonia, hopelessness, negative self-concept)
- **Anxiety predictors:** *Racing, worried, panic, dread, nervous, overthinking* capture the hyperarousal and cognitive distortion characteristic of Generalized Anxiety Disorder
- **Normal predictors:** *Happy, grateful, productive, amazing, excited* indicate positive affect, engagement, and social connection
- **Sentiment features** contribute meaningfully — polarity is a strong discriminating signal between Normal and Depression

---
## 💾 14. Save Best Model

The best model and all necessary artifacts are saved so the pipeline can be deployed or reloaded without retraining.

In [ ]:
# ============================================================
# SECTION 14: SAVE BEST MODEL
# ============================================================

OUTPUT_DIR = '/kaggle/working/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Save best model ---
model_path = os.path.join(OUTPUT_DIR, 'best_model.pkl')
joblib.dump(best_result['model'], model_path)
print(f"✅ Best model saved     : {model_path}")

# --- Save TF-IDF vectorizer ---
tfidf_path = os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.pkl')
joblib.dump(tfidf, tfidf_path)
print(f"✅ TF-IDF vectorizer saved : {tfidf_path}")

# --- Save SVD (needed for GB/XGB models) ---
svd_path = os.path.join(OUTPUT_DIR, 'svd_transformer.pkl')
joblib.dump(svd, svd_path)
print(f"✅ SVD transformer saved   : {svd_path}")

# --- Save label encoder ---
le_path = os.path.join(OUTPUT_DIR, 'label_encoder.pkl')
joblib.dump(le, le_path)
print(f"✅ Label encoder saved     : {le_path}")

# --- Save label names ---
label_names_path = os.path.join(OUTPUT_DIR, 'label_names.txt')
with open(label_names_path, 'w') as f:
    for i, lbl in enumerate(LABEL_NAMES):
        f.write(f"{i}\t{lbl}\n")
print(f"✅ Label names saved       : {label_names_path}")

print("\n📦 All artifacts saved successfully!")
print("\n--- Reloading artifacts (verification) ---")
loaded_model = joblib.load(model_path)
loaded_tfidf = joblib.load(tfidf_path)
print(f"✅ Reload verified: {type(loaded_model).__name__}")

---
## 🔮 15. Prediction Function

We create a clean, reusable `predict_mental_health()` function that handles the full preprocessing → feature extraction → prediction pipeline.

In [ ]:
# ============================================================
# SECTION 15: PREDICTION FUNCTION
# ============================================================

# Determine if best model needs SVD reduction
NEEDS_SVD = best_result['name'] in ['Gradient Boosting', 'XGBoost']

def predict_mental_health(text: str, verbose: bool = True) -> dict:
    """
    Predict the mental health category from input text.
    
    Parameters
    ----------
    text : str
        Raw input text (social media post, message, etc.)
    verbose : bool
        Whether to print a formatted result to stdout.
    
    Returns
    -------
    dict with keys:
        - predicted_label: str
        - probabilities: dict {label: float}
        - polarity: float
        - subjectivity: float
    """
    # Step 1: Preprocess
    cleaned = preprocess_text(text)
    
    # Step 2: TF-IDF
    tfidf_feats = loaded_tfidf.transform([cleaned])
    
    # Step 3: Sentiment features
    blob = TextBlob(text)
    pol, subj = blob.sentiment.polarity, blob.sentiment.subjectivity
    
    # Step 4: Linguistic features
    ling = compute_linguistic_features(text)
    manual_feat = np.array([[pol, subj] + list(ling.values())])
    
    # Step 5: Combine features
    combined = hstack([tfidf_feats, csr_matrix(manual_feat)])
    
    # Step 6: Optionally apply SVD (for GB/XGB)
    if NEEDS_SVD:
        combined = svd.transform(combined)
    
    # Step 7: Predict
    pred_idx = loaded_model.predict(combined)[0]
    pred_label = LABEL_NAMES[pred_idx]
    
    # Step 8: Get probabilities
    try:
        probs = loaded_model.predict_proba(combined)[0]
        prob_dict = {LABEL_NAMES[i]: round(float(p), 4) for i, p in enumerate(probs)}
    except AttributeError:
        prob_dict = {lbl: (1.0 if lbl == pred_label else 0.0) for lbl in LABEL_NAMES}
    
    result = {
        'predicted_label': pred_label,
        'probabilities': prob_dict,
        'polarity': round(pol, 4),
        'subjectivity': round(subj, 4)
    }
    
    if verbose:
        print("\n" + "─" * 55)
        print(f"  📝 Input   : {text[:100]}..." if len(text) > 100 else f"  📝 Input   : {text}")
        print(f"  🎯 Prediction: [{pred_label}]")
        print(f"  📊 Probabilities:")
        for lbl, p in sorted(prob_dict.items(), key=lambda x: -x[1]):
            bar = '█' * int(p * 20)
            print(f"     {lbl:<15} {bar:<20} {p:.2%}")
        print(f"  💬 Polarity: {pol:+.3f}  |  Subjectivity: {subj:.3f}")
        print("─" * 55)
    
    return result


print("✅ predict_mental_health() function defined.")
print("\n=" * 55)
print("RUNNING EXAMPLE PREDICTIONS")
print("=" * 55)

In [ ]:
# --- Example Predictions ---

test_cases = [
    "I feel hopeless and empty. Nothing makes me happy anymore.",
    "I had an amazing day with friends. Feeling grateful and alive!",
    "My heart is racing and I can't stop worrying about everything."
]

for text in test_cases:
    predict_mental_health(text, verbose=True)

---
## 🌍 16. Real-World Applications

This NLP pipeline has broad applicability across mental health support ecosystems:

### 1. 🏥 Mental Health Monitoring Platforms
Integration with digital mental health apps (e.g., journaling apps, therapy platforms) to proactively surface risk signals and alert clinicians to at-risk users.

### 2. 🎓 Educational Institutions
Universities and schools can analyze anonymized student forum posts or help-line messages to identify students who may benefit from counseling services, enabling early intervention.

### 3. 🌐 Online Community Moderation
Reddit, Facebook Groups, and Discord servers can use such models to:
- Automatically flag posts that may require moderator or mental health responder attention
- Route crisis messages to appropriate resources
- Provide users with automated mental health resource suggestions

### 4. 🔬 Research Support
Researchers can use this pipeline to:
- Analyze large-scale longitudinal mental health datasets
- Study temporal trends in mental health discourse
- Measure treatment response through pre/post therapy text analysis

### 5. 📱 Chatbot Integration
Mental health chatbots (e.g., Woebot, Wysa) can use sentiment and classification modules to adapt conversation style and provide appropriate resources based on detected emotional states.

---

## ⚖️ 17. Ethical Considerations

The deployment of mental health AI systems carries profound ethical responsibilities:

### 🔒 Privacy & Consent
- All data collection must be fully anonymized and consent-based
- GDPR, HIPAA, and regional data protection laws must be strictly followed
- Users must be informed if their text is being analyzed

### 📉 Bias & Fairness
- Models trained primarily on English-language Western social media may not generalize across languages, cultures, or demographics
- Systematic bias evaluation (by gender, ethnicity, age) is essential before deployment
- Disparate error rates across groups must be monitored and mitigated

### ⚠️ False Positives & False Negatives
| Error Type | Consequence |
|---|---|
| False Positive | Unnecessary alarm, stigmatization, resource waste |
| False Negative | Missed at-risk individual — potentially life-threatening |

In healthcare settings, **false negatives are far more costly** — recall (sensitivity) should be optimized over precision.

### 👨‍⚕️ Human Oversight — Non-Negotiable
- This tool must operate as a **clinical decision support system**, never a replacement for licensed mental health professionals
- Every flag must be reviewed by a trained human
- Final diagnostic and treatment decisions rest entirely with qualified clinicians

### 🚫 Anti-Surveillance Principle
- This technology must never be used for mass psychological surveillance, government profiling, or corporate exploitation
- Deployment must be governed by transparent policies with independent oversight

---

## 🚀 18. Future Improvements

This project establishes a solid classical NLP baseline. The roadmap for improvement includes:

### 🧠 Advanced Language Models

| Model | Why Valuable |
|---|---|
| **BERT** | Bidirectional context; captures nuanced meaning |
| **RoBERTa** | Optimized BERT; more robust representations |
| **DistilBERT** | Lightweight transformer; suitable for production |
| **Mental-RoBERTa** | Domain-specific fine-tuned model for mental health text |
| **GPT-4 / Claude** | Zero-shot and few-shot classification with prompting |

### 📊 Better Evaluation
- K-fold cross-validation for more robust metric estimates
- Bootstrap confidence intervals for performance metrics
- Clinical validation against gold-standard psychiatric assessments

### 🌐 Multilingual Support
- Fine-tune multilingual models (mBERT, XLM-R) to support non-English communities
- Translate training data and build cross-lingual evaluation benchmarks

### 🔬 Explainable AI (XAI)
- LIME (Local Interpretable Model-Agnostic Explanations)
- SHAP (SHapley Additive exPlanations)
- Attention visualization for transformer models

### ⚡ Real-Time Deployment
- REST API (FastAPI / Flask)
- Docker containerization
- Streamlit / Gradio demo interface
- Kafka stream processing for real-time social media monitoring

### 📈 Extended Classification
- Multi-label classification (comorbid conditions)
- Severity scoring (mild / moderate / severe)
- Temporal trend analysis (worsening vs. improving over time)

---

## 📝 19. Conclusion

### Summary

This notebook presents a **complete, production-quality NLP pipeline** for detecting mental health signals (Depression, Anxiety, Normal) from social media text. We successfully:

✅ Performed comprehensive exploratory data analysis with **15+ visualizations**  
✅ Built a multi-step text preprocessing pipeline  
✅ Engineered **three complementary feature types** (TF-IDF, Sentiment, Linguistic)  
✅ Trained and compared **5 diverse ML classifiers**  
✅ Identified the best model through a rigorous, ranked leaderboard  
✅ Conducted error analysis to understand model limitations  
✅ Provided feature-level explainability  
✅ Saved reusable model artifacts for deployment  
✅ Built a clean, callable `predict_mental_health()` function  

### Key Findings

| Finding | Insight |
|---|---|
| TF-IDF bigrams outperform unigrams | Phrase-level features are more discriminative |
| Sentiment polarity is highly informative | Depression texts have strongly negative polarity |
| Depression-Anxiety boundary is hardest | These classes share substantial linguistic overlap |
| Logistic Regression is competitive | Interpretable models are often sufficient for text tasks |

### Closing Thoughts

Mental health is one of the most pressing public health challenges of our time. NLP offers unprecedented tools to bridge the gap between those who need support and those who can provide it — but only when developed and deployed with **rigor, humility, and deep ethical care**.

This project is a foundation. The real impact lies ahead — in clinical validation, responsible deployment, and ultimately, in the lives that can be reached earlier and supported more effectively.

---

```
╔═══════════════════════════════════════════════════════════╗
║                                                           ║
║        Project Developed By Aastha Mahajan               ║
║                                                           ║
║   GitHub: https://github.com/aastha-m22/                 ║
║           mental-health-nlp-analyzer                     ║
║                                                           ║
║   Domain : NLP · Machine Learning · Mental Health        ║
║   Tools  : Python · sklearn · NLTK · TextBlob · XGBoost  ║
║                                                           ║
╚═══════════════════════════════════════════════════════════╝
```

> *"Data science in service of human well-being is not just a career path — it's a responsibility."*

---

**If this notebook was helpful, please consider:**
- ⬆️ Upvoting this notebook on Kaggle
- ⭐ Starring the GitHub repository
- 💬 Leaving comments with suggestions or questions

---

In [ ]:
# ============================================================
# FINAL SUMMARY REPORT
# ============================================================

print("\n" + "═" * 65)
print("         MENTAL HEALTH NLP ANALYZER — FINAL REPORT")
print("═" * 65)
print(f"  Dataset Size      : {df.shape[0]:,} samples")
print(f"  Features          : TF-IDF ({tfidf.max_features:,}) + Sentiment + Linguistic")
print(f"  Models Trained    : {len(all_results)}")
print(f"  Best Model        : {best_result['name']}")
print(f"  Best F1 Score     : {best_result['f1']:.4f}")
print(f"  Best Accuracy     : {best_result['accuracy']:.4f}")
print(f"  Test Set Size     : {len(y_test):,} samples")
print(f"  Classes           : {', '.join(LABEL_NAMES)}")
print("─" * 65)
print("  Model Leaderboard:")
for _, row in leaderboard.iterrows():
    print(f"   {row['Rank']}. {row['Model']:<28} F1={row['F1 Score']:.4f}  Acc={row['Accuracy']:.4f}")
print("─" * 65)
print("  Saved Artifacts:")
print("    ✅ best_model.pkl")
print("    ✅ tfidf_vectorizer.pkl")
print("    ✅ label_encoder.pkl")
print("    ✅ label_names.txt")
print("═" * 65)
print("\n  Project Developed By Aastha Mahajan")
print("  GitHub: https://github.com/aastha-m22/mental-health-nlp-analyzer")
print("═" * 65)